In [2]:
import analysis
import algorithms
import writing
import pandas as pd

load grain analysis

In [3]:
path =  "..\..\corpus\\metro_sample_1"
sr = 48000
analyzer = analysis.AnalyzerObject(path, sr)
grain_duration = 0.1
df = analyzer.compute_grain_descriptors(grain_duration)
analyzer.save_metadata(df)

Length y: 4499456 Length descr y: 4499456
Length y: 4499456 Length descr y: 4499456
Length y: 4499456 Length descr y: 4499456
Length y: 4499456 Length descr y: 4499456
Length y: 4499456 Length descr y: 4499456
Length y: 4499456 Length descr y: 4499456
Length y: 4499456 Length descr y: 4499456
Saved to csv to: ..\..\corpus\metro_sample_1\metadata\grain_metadata_56d3b567.csv


In [8]:
n_clusters = 3
metadata_path = "..\..\corpus\\metro_sample_1\metadata\grain_metadata_56d3b567.csv"
df, df_scaled = analyzer.scale_metadata(metadata_path)#, n_clusters=n_clusters)
# df[["flatness", "kurtosis"]]
df_scaled[["centroid", "flux", "rolloff", "flatness", "spread", "skewness", "kurtosis"]]

,centroid,flux,rolloff,flatness,spread,skewness,kurtosis
0,-2.505826,-0.571721,-2.897352,-2.318562,-3.899694,-2.718013,-1.626665
1,0.227044,-0.571721,-1.489157,2.576634,-2.125044,-2.751193,-1.529627
2,2.088192,-0.568247,1.334625,3.201929,1.667376,-1.297238,-1.032166
3,2.203700,-0.551090,1.320781,1.290035,1.995511,-1.480386,-1.144836
4,1.373039,-0.538189,1.269034,0.756738,1.049614,-1.041374,-0.908480
...,...,...,...,...,...,...,...
932,-1.323012,-0.167828,-1.888820,-1.464859,-2.036325,1.793807,2.653502
933,-1.341095,-0.217456,-1.863417,-1.389259,-1.966301,1.876595,2.599406
934,-1.269319,-0.229085,-1.803067,-1.329681,-1.911583,1.696600,2.345170
935,-1.327282,-0.123695,-1.896212,-1.440314,-2.039717,1.874383,2.757242


dict_cluster, kmeans = analyzer.get_dict_cluster

In [9]:
dict_cluster, kmeans = analyzer.get_cluster_dict(metadata_path, n_clusters)

In [10]:
import numpy as np
densities = [58, 104, 93]#, 185, 370]
synthesizer = algorithms.MarkovGranulizer(sr=analyzer.sr, densities=densities)
delta_t = 0.5
n_iterations = 30
seed=45
n_grains = 2
np.random.seed()
init_states = [np.random.randint(0, len(densities)*n_clusters) for _ in range(n_grains)]
output_buffer, params = synthesizer.run(
    analyzer.y,
    n_iterations,
    delta_t=delta_t,
    n_grains=n_grains,
    window=np.hanning,
    n_clusters=n_clusters,
    seed=seed,
    init_states=init_states,
    grains=analyzer.grains(grain_duration),
    dict_clusters=dict_cluster,
)


In [11]:
(params)
params["init_states"] = [int(i) for i in params["init_states"]]
params
output_buffer.shape

(2, 1440000)

In [12]:

writing.save_output_data(output_data=output_buffer.T, sr=analyzer.sr, parametre_dict=params, output_dir=path)